# MCR Stage G — GRPO sanity with 4-term composite reward (Qwen3.6-35B-A3B)

**Goal**: test if MCR-4 (outcome + probe + adversary + commit) lifts accuracy over R_outcome baseline on Qwen3.6-35B-A3B in ~30 steps (G2-scale sanity).

**Reward composition** (per-response, same for all tokens):
```
R_total(y) = R_outcome(y)                                    # from MCQ verifier
           + λ_probe · (σ(LogReg_L11(z_L11_pool)) − 0.5)      # Stage D probe, AUROC 0.78
           + λ_commit · 𝟙[response ended with \boxed AND response_len < max_new]  # Stage compute analysis
           + β_adv · (D_φ(z_concat) − 0.5)                    # Stage F adversary trust bonus
           − β_g · max(0, R_mech − τ) · (1 − D_φ(z_concat))  # Goodhart guard
```
where:
- `z_L11_pool = PCA_64(StandardScaler(mean_pool(residual_L11)))`  [Stage D artifact]
- `z_concat = concat(PCA_64(L11), PCA_64(L17), PCA_64(L23))`  [192-d for D_φ]
- `D_φ` = MLP(192→64→32→1) warm-started from Stage F (val AUROC 0.82)

**Coefficients (sanity pass)**:
- λ_probe = 0.5, λ_commit = 0.3, β_adv = 0.3, β_g = 1.0
- τ = 0.0 (median of probe at warm-start ≈ 0)

**Ablation to run first**: `V4_full` (all 4 terms) vs reference V0 = R_outcome only (= G2 R0 baseline = 40%).

**Expected outcome**:
- V4_full > 43% on 50Q eval → encourages scaling to 200 steps × 3 seeds
- V4_full ≈ 40% → terms conflict or λ wrong, sweep coefficients
- V4_full < 38% → reward shaping broke something, abort

**Budget**: ~2h on RTX PRO 6000 Blackwell 96 GB. Same as G2 R0 run.

**Pre-reqs**:
- `mcr_d_density_models.pkl` on /content (or HF)
- `mcr_f_adversary_warmstart.pt` on /content
- Base Qwen3.6-35B-A3B downloadable from HF

## Cell 1 — Install (same stack as G2 v2 + sklearn for loading Stage D models)

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def _pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check)

def _check():
    try:
        import transformers
        try:
            from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
        except ImportError:
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
        return 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, transformers.__version__
    except Exception as e:
        return False, f'import-error: {e}'

ok, ver = _check()
print(f'Current: transformers={ver}, qwen3_5_moe={ok}')

if not ok:
    print('\u2192 Installing transformers @ main + stack')
    _pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
         'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
         'protobuf', 'scipy', 'peft')
    _pip('uninstall', '-y', 'transformers', check=False)
    for p in sys.path + site.getsitepackages() + [site.getusersitepackages()]:
        tr = Path(p) / 'transformers'
        if tr.exists():
            shutil.rmtree(tr, ignore_errors=True)
    SRC_DIR = '/content/transformers_src'
    if os.path.exists(SRC_DIR):
        shutil.rmtree(SRC_DIR)
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/huggingface/transformers.git', SRC_DIR], check=True)
    _pip('install', '--force-reinstall', '--no-deps', '--no-cache-dir', SRC_DIR)
    _pip('install', '--no-cache-dir', 'flash-linear-attention', check=False)
    for mod in list(sys.modules):
        if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
            del sys.modules[mod]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('\u2705 HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

import json, time, random, re, gc, pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

## Cell 2 — CFG

Run flag `VARIANT` switches between:
- `V0_outcome` — R_outcome only (skip if you already have G2 R0 = 40%)
- `V4_full`    — all 4 terms (the MCR-4 sanity test)

In [ ]:
CFG = dict(
    model_id='Qwen/Qwen3.6-35B-A3B',
    variant='V4_full',               # V0_outcome | V4_full
    # training
    n_steps=30,
    rollouts_per_step=4,
    max_new_tokens=2048,
    lr=3e-6,
    lora_r=32, lora_alpha=64,
    temperature=1.0, top_p=0.95,
    # dataset (reuse G2/Stage B splits logic)
    difficulty_filter=['easy', 'middle'],
    seed=42,
    s4l0_n=100, g1_total=150,  # skip first 250
    train_start=450, train_n=200,    # questions [450:650] for training
    eval_mid_start=650, eval_mid_n=30,
    eval_final_start=680, eval_final_n=50,
    # reward coefficients (sanity defaults)
    lambda_probe=0.5,
    lambda_commit=0.3,
    beta_adv=0.3,
    beta_goodhart=1.0,
    tau_probe=0.0,
    # artifacts
    stage_d_pkl='/content/mcr_d_density_models.pkl',
    stage_f_pt='/content/mcr_f_adversary_warmstart.pt',
)
print(json.dumps(CFG, indent=2))

## Cell 3 — Load Qwen3.6-35B-A3B + LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, PeftModel

tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForImageTextToText.from_pretrained(
    CFG['model_id'], dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True,
)
base_model.eval()
print(f'Base loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# LoRA on attention projections
lora_cfg = LoraConfig(
    r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'], lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM', bias='none',
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

## Cell 4 — Load Stage D (probe) + Stage F (adversary) artifacts

In [ ]:
from huggingface_hub import hf_hub_download

# If artifacts not local, download from HF — adapt if you uploaded them elsewhere
stage_d_path = CFG['stage_d_pkl']
stage_f_path = CFG['stage_f_pt']

if not Path(stage_d_path).exists():
    print(f'\u26a0\ufe0f  {stage_d_path} not found locally. Upload it via Colab file browser, or adjust CFG.')
if not Path(stage_f_path).exists():
    print(f'\u26a0\ufe0f  {stage_f_path} not found locally. Upload it via Colab file browser, or adjust CFG.')

with open(stage_d_path, 'rb') as f:
    models_d = pickle.load(f)
scaler_L11 = models_d['L11']['scaler']
pca_L11    = models_d['L11']['pca']
logreg_L11 = models_d['L11']['logreg']
scaler_L17 = models_d['L17']['scaler']; pca_L17 = models_d['L17']['pca']
scaler_L23 = models_d['L23']['scaler']; pca_L23 = models_d['L23']['pca']

class AdversaryMLP(nn.Module):
    def __init__(self, d_in=192, h1=64, h2=32, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, h1), nn.GELU(), nn.Dropout(p),
            nn.Linear(h1, h2), nn.GELU(), nn.Dropout(p),
            nn.Linear(h2, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

adv_mlp = AdversaryMLP().to('cuda')
adv_ckpt = torch.load(stage_f_path, map_location='cuda')
adv_mlp.load_state_dict(adv_ckpt['state_dict'])
adv_mlp.eval()
print(f'\u2705 Probe (LogReg L11) loaded')
print(f'\u2705 Adversary MLP loaded (warm AUROC: {adv_ckpt["val_auroc"]:.4f})')

## Cell 5 — Dataset (disjoint from Stage B [250:450])

In [ ]:
from datasets import load_dataset
raw = load_dataset('m-a-p/SuperGPQA', split='train')
filtered = raw.filter(lambda ex: ex['difficulty'] in CFG['difficulty_filter'])
shuffled = filtered.shuffle(seed=CFG['seed'])

train_ds = shuffled.select(range(CFG['train_start'], CFG['train_start']+CFG['train_n']))
eval_mid = shuffled.select(range(CFG['eval_mid_start'], CFG['eval_mid_start']+CFG['eval_mid_n']))
eval_final = shuffled.select(range(CFG['eval_final_start'], CFG['eval_final_start']+CFG['eval_final_n']))
print(f'Train: {len(train_ds)}   MidEval: {len(eval_mid)}   FinalEval: {len(eval_final)}')

## Cell 6 — Prompt + verifier + pooling hooks

In [ ]:
def format_prompt(ex):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(ex['options']))
    msgs = [{'role': 'user', 'content':
        f"Answer the following multiple-choice question. Reason step by step, then put the letter of the correct answer in \\boxed{{}}.\n\nQuestion: {ex['question']}\n\nOptions:\n{choices}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')

def extract_letter(completion, n_opts):
    m = list(BOXED_RE.finditer(completion))
    if m:
        letter = m[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts:
            return letter
    tail = completion[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands and ord(cands[-1]) - ord('A') < n_opts:
        return cands[-1]
    return None

def verify(ex, completion):
    pred = extract_letter(completion, len(ex['options']))
    return pred is not None and pred == ex['answer_letter']

def has_boxed(completion):
    return bool(BOXED_RE.search(completion))

def get_layer_module(m, idx):
    candidates = [m]
    if hasattr(m, 'base_model') and m.base_model is not m:
        candidates.append(m.base_model.model if hasattr(m.base_model, 'model') else m.base_model)
    if hasattr(m, 'model'):
        candidates.append(m.model)
    paths = [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]
    for start in candidates:
        for path in paths:
            cur = start; okp = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: okp = False; break
            if okp and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except (IndexError, TypeError): continue
    raise RuntimeError(f'Layer {idx} not found')

class MultiCapture:
    def __init__(self, layers):
        self.h = {i: None for i in layers}
    def make_hook(self, layer_idx):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.h[layer_idx] = h.detach()
        return hook
    def clear(self):
        for k in self.h: self.h[k] = None

def register_hooks(m, layers):
    cap = MultiCapture(layers)
    handles = [get_layer_module(m, i).register_forward_hook(cap.make_hook(i)) for i in layers]
    return cap, handles

print('Prompt + verifier + hook utils ready.')

## Cell 7 — MCR reward computation

Given batch of rollouts (text + response masks + captured hidden states), compute per-rollout components + total.

In [ ]:
def pool_mean(hidden, resp_mask):
    """hidden: [B, T, d].  resp_mask: [B, T] bool. Returns [B, d] mean over response tokens."""
    mask_f = resp_mask.float().unsqueeze(-1)
    pooled = (hidden.float() * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
    return pooled

def pca_transform_np(pooled_np, scaler, pca):
    return pca.transform(scaler.transform(pooled_np))

def compute_mcr_reward(
    completions, outcomes, response_lens, h_L11, h_L17, h_L23, resp_mask,
    variant='V4_full',
):
    """
    completions: list[str] len B
    outcomes: np array of 0/1 len B
    response_lens: list[int] len B
    h_L{11,17,23}: [B, T, d] tensors on CUDA
    resp_mask: [B, T] bool on CUDA
    Returns: dict with components + total (all [B] np arrays)
    """
    B = len(completions)
    # R_outcome (from verifier)
    r_outcome = outcomes.astype(np.float32)

    if variant == 'V0_outcome':
        return {'r_outcome': r_outcome, 'r_probe': np.zeros(B), 'r_commit': np.zeros(B),
                'd_phi': np.full(B, 0.5), 'r_total': r_outcome}

    # Pool L11, L17, L23 per rollout (response tokens only)
    with torch.no_grad():
        p11 = pool_mean(h_L11, resp_mask).cpu().numpy()
        p17 = pool_mean(h_L17, resp_mask).cpu().numpy()
        p23 = pool_mean(h_L23, resp_mask).cpu().numpy()

    # R_probe: LogReg_L11 on PCA(L11)
    z11 = pca_transform_np(p11, scaler_L11, pca_L11)  # [B, 64]
    probe_prob = logreg_L11.predict_proba(z11)[:, 1]  # [B]
    r_probe = probe_prob - 0.5                        # [-0.5, 0.5]

    # R_commit: binary
    r_commit = np.array([
        1.0 if (has_boxed(completions[i]) and response_lens[i] < CFG['max_new_tokens']) else 0.0
        for i in range(B)
    ], dtype=np.float32)

    # D_φ: MLP on concat PCA(L11,L17,L23)
    z17 = pca_transform_np(p17, scaler_L17, pca_L17)
    z23 = pca_transform_np(p23, scaler_L23, pca_L23)
    z_concat = np.concatenate([z11, z17, z23], axis=1).astype(np.float32)  # [B, 192]
    with torch.no_grad():
        d_logits = adv_mlp(torch.from_numpy(z_concat).to('cuda'))
        d_phi = torch.sigmoid(d_logits).cpu().numpy()

    # Combine
    lam_p = CFG['lambda_probe']; lam_c = CFG['lambda_commit']
    bet_a = CFG['beta_adv']; bet_g = CFG['beta_goodhart']
    tau = CFG['tau_probe']

    # Mech score (for Goodhart guard): use r_probe as proxy for R_mech
    r_mech = r_probe  # simple; could extend if R_edges is added later
    goodhart_pen = bet_g * np.maximum(0, r_mech - tau) * (1 - d_phi)

    r_total = (
        r_outcome
        + lam_p * r_probe
        + lam_c * r_commit
        + bet_a * (d_phi - 0.5)
        - goodhart_pen
    )
    return {
        'r_outcome': r_outcome, 'r_probe': r_probe, 'r_commit': r_commit,
        'd_phi': d_phi, 'goodhart_pen': goodhart_pen, 'r_total': r_total.astype(np.float32),
    }

print('MCR reward function ready.')

## Cell 8 — GRPO single-step

Generates 4 rollouts per question, computes MCR reward, advantages, backward step. Uses disable_adapter() for ref log-probs to save VRAM (~8GB saved vs duplicate ref model).

In [ ]:
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG['lr'])

def grpo_step(ex, variant):
    prompt = format_prompt(ex)
    enc = tok(prompt, return_tensors='pt').to('cuda')
    P = enc.input_ids.shape[1]
    pad_id = tok.pad_token_id; eos_id = tok.eos_token_id

    # 1) Generate B rollouts
    with torch.no_grad():
        gen = model.generate(
            **enc, max_new_tokens=CFG['max_new_tokens'],
            do_sample=True, temperature=CFG['temperature'], top_p=CFG['top_p'],
            num_return_sequences=CFG['rollouts_per_step'],
            pad_token_id=pad_id, use_cache=True,
        )
    B = gen.shape[0]; T = gen.shape[1]

    # Determine per-sequence response lengths
    resp_lens = []
    for b in range(B):
        row = gen[b, P:]
        L = 0
        for tid in row.tolist():
            if tid == eos_id or tid == pad_id: break
            L += 1
        if L < len(row) and row[L].item() == eos_id:
            L += 1
        resp_lens.append(L)
    resp_mask = torch.zeros(B, T, dtype=torch.bool, device='cuda')
    for b in range(B):
        resp_mask[b, P:P+resp_lens[b]] = True

    # 2) Verifier + completion text
    completions = [tok.decode(gen[b, P:P+resp_lens[b]], skip_special_tokens=True) for b in range(B)]
    outcomes = np.array([int(verify(ex, c)) for c in completions])

    # 3) Forward pass (with hooks) to get hidden states + log-probs
    attn = (gen != pad_id).long()
    cap, handles = register_hooks(model, [11, 17, 23])
    try:
        out = model(input_ids=gen, attention_mask=attn, use_cache=False)
    finally:
        for h in handles: h.remove()
    logits = out.logits  # [B, T, V]
    logp = F.log_softmax(logits.float(), dim=-1)
    # Shift for next-token log-prob
    logp_tokens = logp[:, :-1, :].gather(2, gen[:, 1:].unsqueeze(-1)).squeeze(-1)  # [B, T-1]
    resp_mask_shift = resp_mask[:, 1:]  # align with logp_tokens

    # 4) Reference log-probs via disable_adapter()
    with torch.no_grad(), model.disable_adapter():
        ref_out = model(input_ids=gen, attention_mask=attn, use_cache=False)
        ref_logp = F.log_softmax(ref_out.logits.float(), dim=-1)
        ref_logp_tokens = ref_logp[:, :-1, :].gather(2, gen[:, 1:].unsqueeze(-1)).squeeze(-1)

    # 5) Compute MCR reward
    rewards = compute_mcr_reward(
        completions, outcomes, resp_lens,
        cap.h[11], cap.h[17], cap.h[23], resp_mask,
        variant=variant,
    )
    r_total = torch.from_numpy(rewards['r_total']).to('cuda')

    # 6) Group-relative advantages
    adv = (r_total - r_total.mean()) / (r_total.std() + 1e-8)  # [B]
    adv_per_token = adv.unsqueeze(-1).expand(-1, logp_tokens.shape[1])

    # 7) PG loss + KL
    pg_loss = -(logp_tokens * adv_per_token * resp_mask_shift.float()).sum() / resp_mask_shift.float().sum().clamp(min=1)
    kl_per_token = (logp_tokens - ref_logp_tokens) * resp_mask_shift.float()
    kl = kl_per_token.sum() / resp_mask_shift.float().sum().clamp(min=1)
    loss = pg_loss + 0.04 * kl

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    optimizer.step()

    return {
        'loss': loss.item(), 'pg_loss': pg_loss.item(), 'kl': kl.item(),
        'outcome_mean': outcomes.mean(), 'r_total_mean': r_total.mean().item(),
        'probe_mean': rewards['r_probe'].mean(), 'commit_mean': rewards['r_commit'].mean(),
        'd_phi_mean': rewards['d_phi'].mean(),
        'tok_mean': np.mean(resp_lens),
    }

## Cell 9 — Eval helper

In [ ]:
@torch.no_grad()
def eval_sgpqa(dataset, label):
    model.eval()
    correct = 0; total = 0
    from tqdm.auto import tqdm
    for i, ex in enumerate(tqdm(dataset, desc=label)):
        prompt = format_prompt(ex)
        enc = tok(prompt, return_tensors='pt').to('cuda')
        gen = model.generate(
            **enc, max_new_tokens=CFG['max_new_tokens'],
            do_sample=False, pad_token_id=tok.pad_token_id,
        )
        P = enc.input_ids.shape[1]
        completion = tok.decode(gen[0, P:], skip_special_tokens=True)
        correct += int(verify(ex, completion))
        total += 1
        if (i+1) % 10 == 0:
            print(f'  [{label}] {total}/{len(dataset)}: running acc={100*correct/total:.1f}%')
    acc = correct / total
    print(f'{label} FINAL: {100*acc:.2f}%')
    model.train()
    return acc

## Cell 10 — Train loop

30 steps, mid-eval at step 15, final eval at end. Logs per-term reward values to catch misbehavior early.

In [ ]:
print(f'{"="*70}')
print(f'Running {CFG["variant"]}  (LR={CFG["lr"]}, rollouts/step={CFG["rollouts_per_step"]})')
print(f'{"="*70}')

log_lines = []
t0 = time.time()
step_idx = 0
train_iter = iter(train_ds)

for step in range(1, CFG['n_steps']+1):
    try:
        ex = next(train_iter)
    except StopIteration:
        train_iter = iter(train_ds); ex = next(train_iter)
    st = time.time()
    info = grpo_step(ex, CFG['variant'])
    dt = time.time() - st
    line = (f'[{CFG["variant"]} {step:2d}/{CFG["n_steps"]}] '
            f'loss={info["loss"]:+.4f}  outcome={info["outcome_mean"]:.2f}  '
            f'probe={info["probe_mean"]:+.3f}  commit={info["commit_mean"]:.2f}  '
            f'D_phi={info["d_phi_mean"]:.3f}  R={info["r_total_mean"]:+.3f}  '
            f'tok={info["tok_mean"]:.0f}  kl={info["kl"]:+.4f}  {dt:.0f}s')
    print(line); log_lines.append(line)

    if step == CFG['n_steps'] // 2:
        mid_acc = eval_sgpqa(eval_mid, f'{CFG["variant"]}_mid@{step}')
        log_lines.append(f'MID_EVAL @ step {step}: {100*mid_acc:.2f}%')

print(f'\n{"="*70}')
print(f'[{CFG["variant"]}] Final eval (n={len(eval_final)})...')
print(f'{"="*70}')
final_acc = eval_sgpqa(eval_final, f'{CFG["variant"]}_final')
log_lines.append(f'FINAL: {100*final_acc:.2f}%')

total_wall = (time.time() - t0) / 60
print(f'\nWall time: {total_wall:.1f} min')
print(f'\nBaseline reference (G2 R0 on same protocol): 40.0%')
print(f'{CFG["variant"]} final:  {100*final_acc:.2f}%')
print(f'\u0394 vs baseline:  {100*(final_acc - 0.40):+.2f}pp')

# Save log + checkpoint
out_path = f'/content/mcr_g_sanity_{CFG["variant"]}.json'
with open(out_path, 'w') as f:
    json.dump({
        'variant': CFG['variant'], 'final_acc': float(final_acc),
        'baseline': 0.40, 'delta_pp': float(100*(final_acc - 0.40)),
        'n_steps': CFG['n_steps'], 'wall_min': total_wall, 'log': log_lines,
    }, f, indent=2)
print(f'\u2705 Saved: {out_path}')

## Interpretation

- `final_acc ≥ 43%` → MCR-4 ships; scale to 200 steps × 3 seeds + isolate V2/V3 ablations.
- `40% ≤ final_acc < 43%` → within-noise; sweep λ coefficients (probe/commit/adv).
- `final_acc < 40%` → reward shaping broke something, investigate via per-term logs: is probe_mean drifting negative? is D_phi saturating? is commit pulling policy to minimal responses?